In [1]:
# ** This cell is needed since we are not in the src directory **
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx16g -Xms8g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # SPARK DIRECTORY FOR THREADS / PERSIST
    
    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \
    
    # THIS IS WHERE SPARK WILL PUT ITS TEMPERARY VARIABLES 
    # .config("spark.local.dir", os.path.expanduser("~/external-spark-tmp"))

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "512") \

    # prevent breaking pipes 
    .config("spark.reducer.maxReqsInFlight", "1") \
    .config("spark.shuffle.io.preferDirectBufs", "false") \
    .config("spark.shuffle.file.buffer", "32k") \
    
    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
25/09/06 18:00:09 WARN Utils: Your hostname, Mac-mini.local resolves to a loopback address: 127.0.0.1; using 192.168.1.172 instead (on interface en1)
25/09/06 18:00:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/06 18:00:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


In [7]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler

In [8]:
 parquet_direcotry = "/Volumes/CrucialX6/Home/Desktop/eeg-ds004504-BACKUP/notebooks"

In [9]:
alz_df_spark = spark.read.parquet(f"{parquet_direcotry}/features_alz_extra_features_Apr19_2141.parquet")
cntrl_df_spark = spark.read.parquet(f"{parquet_direcotry}/features_cntrl_extra_features_Apr19_2141.parquet")

In [10]:
# rename for useability, I wanted it to be clear that the dataframes are spark.sql types when we create them above
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [11]:
alz_df.show()

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|  7.020328E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|  3.399499E-4|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power|  0.087231696|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power| 0.0011391409|      band|
|  sub-008|   ep-0|      Fp1| custom1|      Power| 4.2795436E-4|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy|   0.34805238| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower|  0.011235955| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power| 0.0014689578|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|  5.043481E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power|   0.08462298|      band|
|  sub-008| 

# Data Processing and Dimensionality Reduction

In [12]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [28]:
full_df = alz_df.unionByName(cntrl_df)

In [29]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")

In [30]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


25/09/06 18:06:15 WARN CacheManager: Asked to cache already cached data.
25/09/06 18:06:15 WARN CacheManager: Asked to cache already cached data.
25/09/06 18:06:15 WARN CacheManager: Asked to cache already cached data.


In [31]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [32]:
from functools import reduce



full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()

25/09/06 18:06:17 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

In [33]:
full_df.repartition(16).persist()

25/09/06 18:06:17 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

In [34]:

from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide
feature_cols = [c for c in full_df.columns if c not in ("SubjectID", "EpochID", "label")]


# ! Checking how this affects ML performance!
# full_df = normalize_by_column_per_subject_wide(full_df, feature_cols)
full_df, _ = min_max_normalize(full_df, full_df, feature_cols)



full_df.repartition(16).persist()
# print("finished normalizing by column per subject") # old 
print("fnished normalizing across the dataset")

finished normalizing by column per subject


In [35]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(full_df.drop("label"), pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")

PCA model fitted with 22 components to capture 95% variance


In [36]:
pca_model.explainedVariance

DenseVector([0.5299, 0.1444, 0.0766, 0.0359, 0.0304, 0.0256, 0.022, 0.0133, 0.0111, 0.01, 0.008, 0.0071, 0.0059, 0.0049, 0.0044, 0.0039, 0.0037, 0.0034, 0.0029, 0.0028, 0.0025, 0.0024])

In [37]:
import pandas as pd
import numpy as np

# Convert DenseMatrix to NumPy
pc_matrix = np.array(pca_model.pc.toArray())  # shape: (n_features, n_components)

# Create DataFrame of loadings
loadings_df = pd.DataFrame(pc_matrix, index=pca_input_cols, columns=[f"PC{val}" for val in range(1, k_val+1)])

# Get top 10 features for each component by absolute contribution
for pc in loadings_df.columns:
    print(f"\nTop features contributing to {pc}:")
    display(loadings_df[pc].abs().sort_values(ascending=False).head(10))



Top features contributing to PC1:


C3_TotalEnergy    0.179431
C4_TotalEnergy    0.172404
P4_TotalEnergy    0.164052
Fz_TotalEnergy    0.163908
T6_TotalEnergy    0.163092
T3_TotalEnergy    0.163059
T4_TotalEnergy    0.162530
P3_TotalEnergy    0.162445
O2_TotalEnergy    0.161164
F3_TotalEnergy    0.160757
Name: PC1, dtype: float64


Top features contributing to PC2:


C3_TotalEnergy    0.187642
C4_TotalEnergy    0.180753
Cz_TotalEnergy    0.170561
F3_TotalEnergy    0.169284
F4_TotalEnergy    0.168033
Fz_TotalEnergy    0.165370
O2_Delta_Power    0.163456
T4_TotalEnergy    0.162560
T3_TotalEnergy    0.159703
F7_TotalEnergy    0.158488
Name: PC2, dtype: float64


Top features contributing to PC3:


...

F7_Delta_Power      0.288491
T5_Theta_Power      0.253349
O2_TotalEnergy      0.250222
T5_TotalEnergy      0.207203
T3_TotalEnergy      0.193482
T6_custom1_Power    0.168642
T3_Alpha_Power      0.167800
F7_Alpha_Power      0.164879
T3_custom1_Power    0.164202
P4_Theta_Power      0.161071
Name: PC20, dtype: float64


Top features contributing to PC21:


Pz_Theta_Power    0.257184
O2_TotalEnergy    0.247740
Cz_Theta_Power    0.231538
T3_Theta_Power    0.205039
T6_Theta_Power    0.202040
P3_Theta_Power    0.195769
O2_Theta_Power    0.189211
Fz_TotalEnergy    0.181848
T4_Theta_Power    0.176539
Cz_Alpha_Power    0.168777
Name: PC21, dtype: float64


Top features contributing to PC22:


O2_TotalEnergy     0.342490
T6_TotalEnergy     0.299430
Fp1_Delta_Power    0.234594
T4_TotalEnergy     0.224378
T6_Theta_Power     0.208760
Fp2_TotalEnergy    0.201229
O2_Delta_Power     0.183345
T6_Delta_Power     0.173567
O2_Alpha_Power     0.160023
P4_TotalEnergy     0.155616
Name: PC22, dtype: float64

In [38]:
from dimensionality_reduction import apply_pca_model
full_df = apply_pca_model(full_df, pca_input_cols, pca_model, k_val)

In [39]:
from dimensionality_reduction import min_max_normalize_post_pca_by_subject

# Yes we normalize two times on prupose two different ways on purpose
full_df = min_max_normalize_post_pca_by_subject(full_df)


In [40]:
full_df = full_df.toPandas()

In [41]:
# Machine Learning results are below 

In [42]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def subject_knn_recall(df, max_subjects=10, test_size=0.3):
    # Make sure features are proper numpy arrays
    df = df.copy()
    df['features'] = df['features'].apply(lambda x: np.array(x))
    
    # Extract feature matrix
    X = np.stack(df['features'].values)
    y = df['SubjectID'].values
    df['SubjectID'] = y  # just to be sure they're strings

    subjects = df['SubjectID'].unique()
    np.random.shuffle(subjects)

    results = []

    for n_subjects in range(2, min(max_subjects + 1, len(subjects) + 1)):
        selected = subjects[:n_subjects]
        df_sel = df[df['SubjectID'].isin(selected)]

        # Split train/test *per subject*
        train_list, val_list = [], []
        for subj in selected:
            subj_df = df_sel[df_sel['SubjectID'] == subj]
            train, val = train_test_split(subj_df, test_size=test_size, random_state=42)
            train_list.append(train)
            val_list.append(val)

        train_df = pd.concat(train_list)
        val_df = pd.concat(val_list)

        X_train = np.stack(train_df['features'].values)
        y_train = train_df['SubjectID'].values

        X_val = np.stack(val_df['features'].values)
        y_val = val_df['SubjectID'].values

        clf = KNeighborsClassifier(n_neighbors=1, weights='uniform', metric='minkowski', p=2)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)

        recall = recall_score(y_val, y_pred, average='macro')  # You could also try 'micro'
        results.append((n_subjects, recall))
        print(f"{n_subjects} subjects — Recall (macro): {recall:.4f}")

    return pd.DataFrame(results, columns=["Num_Subjects", "Recall"])

# Example run:
recall_df = subject_knn_recall(full_df, max_subjects=65, test_size=0.20)


2 subjects — Recall (macro): 1.0000
3 subjects — Recall (macro): 1.0000
4 subjects — Recall (macro): 1.0000
5 subjects — Recall (macro): 0.9973
6 subjects — Recall (macro): 0.9962
7 subjects — Recall (macro): 0.9958
8 subjects — Recall (macro): 0.9961
9 subjects — Recall (macro): 0.9961
10 subjects — Recall (macro): 0.9955
11 subjects — Recall (macro): 0.9959
12 subjects — Recall (macro): 0.9955
13 subjects — Recall (macro): 0.9945
14 subjects — Recall (macro): 0.9946
15 subjects — Recall (macro): 0.9943
16 subjects — Recall (macro): 0.9946
17 subjects — Recall (macro): 0.9949
18 subjects — Recall (macro): 0.9952
19 subjects — Recall (macro): 0.9940
20 subjects — Recall (macro): 0.9939
21 subjects — Recall (macro): 0.9929
22 subjects — Recall (macro): 0.9923
23 subjects — Recall (macro): 0.9926
24 subjects — Recall (macro): 0.9918
25 subjects — Recall (macro): 0.9920
26 subjects — Recall (macro): 0.9917
27 subjects — Recall (macro): 0.9913
28 subjects — Recall (macro): 0.9914
29 subjec